# Exploratory Data Analysis (EDA) & Data Preparation

**Dataset:** Credit Score of customers with 8 consecutive month period data (`credit_score.csv`, Kaggle Dataset)
**Target variable:** `Credit_Score` (28 features)


## Dataset Overview

**What it is**. 

**Reason of choice**.

**EDA pipeline**.

**Reproducibility**.

## Download dataset

Uncomment this if the dataset is not downloaded in the folder

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("ayushsharma0812/dataset-for-credit-score-classification")

# print("Path to dataset files:", path)

## Setup
Import packages, set one consistent plot style and define constant reused across the notebook

In [ ]:
# Setup: Libraries and global plot style, and constants
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# One consistent, readable style for every figure

sns.set_theme(style='whitegrid', context="notebook", font_scale = 1.0)
plt.rcParams.update({
    "figure.figsize":(10,6),
    "figure.dpi":100,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 12
})

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)
print("Setup completed successfully.")

## Load datasets

In [ ]:
file_path = Path(r"./Dataset/Credit_score_cleaned_data.csv")
if not file_path.exists():
    raise FileNotFoundError(
        f"{file_path} not found. Download it from commented above codeblock from kaggle."
    )
df = pd.read_csv(file_path)
df.columns = df.columns.str.lower()

# Set options to display full columns
pd.set_option('display.max_columns', None)

In [ ]:
print(f"Dataset shape: {df.shape[0]:,} row x {df.shape[1]} columns")
print(f"Dataset Overview: {df.info()}")
print(f"First 5 rows: {df.head(5)}")

## Data types and data dictionary

The table below documents both the storage type and the real meaning of each column.

In [ ]:
data_dictionary = {

}
dtype_table = pd.DataFrame({
    "Pandas dtype": df.dtypes.astype(str),
    "Unique values": df.nunique(),
    "Variable type": [data_dictionary[c][0] for c in df.columns],
    "Meaning/ Unit": [data_dictionary[c][1] for c in df.columns]
})
dtype_table

In [ ]:
# Anomolies in the minimum of total_emi_per_month (should above zero)

df = df[df['total_emi_per_month' > 0]]
df.describe()

## Basic statistical summary

In [ ]:
# Basic statistical summary - numerical columns
df.describe().T.round(3)

In [ ]:
# Basic statistical summary - categorical columns
cat_summary = df.dtype(exclude=np.number).descibe().T
cat_summary["top %"] = (cat_summary["freq"].astype(int) / len(df) * 100).round(1)
cat_summary

## Inital missing value check

In [ ]:
print("Total missing cells:", int(df.isnull().sum().sum()))
df.isnull().sum().to_frame("Missing count")

## Overall about Observations


#### Explore about qualitative variables

In [ ]:
cross_tab_per_ocp = pd.crosstab(index = df['occupation'], columns = df['credit_score'],
                            normalize = 'columns')[['Poor', 'Standard', 'Good']]*100
sns.heatmap(data = cross_tab_per_ocp, annot = True, cmap = 'Blues', vmin = 4, vmax = 8, fmt = '.1f', linewidth = 0.5)
plt.title("Percentage of Occupations across Credit Score groups")
plt.show()

In [ ]:
df_prop = df.groupby("payment_of_min_amount")['credit_score'].value_counts(normalize=True).rename('proportion').reset_index()
df_prop['percentage'] = df_prop['proportion']*100
plt.figure(figsize = (6,5))
colors = {'Standard': 'Blue', 'Good': 'Green', 'Poor': 'Red'}
ax = sns.barplot(data = df_prop, x = 'payment_of_min_amount', y = 'percentage', hue = 'credit_score', palette = colors)
for container in ax.containers:
      ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=9)

plt.title("Proportion of Credit Score by Payment of Min Amount")
plt.xlabel("Payment of Min Amount")
plt.ylabel("Percentage (%)")
plt.legend(title = 'Credit Score', bbox_to_anchor = (1.05,1), loc = 'upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Add 2 more columns for payment_behaviour
df['payment_level'] = 'default'
df.loc[df['payment_behaviour'].str.contains('Low', case = False, na = False), 'payment_level'] = 'Low payment'
df.loc[df['payment_behaviour'].str.contains('High',case = False, na = False), 'payment_level'] = 'High payment'

df['payment_volume'] = 'default'
df.loc[df['payment_behaviour'].str.contains('Small', case = False, na = False), 'payment_volume'] = 'Small payment'
df.loc[df['payment_behaviour'].str.contains('Medium', case = False, na = False), 'payment_volume'] = 'Medium payment'
df.loc[df['payment_behaviour'].str.contains('Large', case = False, na = False), 'payment_volume'] = 'Large payment'

# Setup layout for ordinal variables
ordinal_cols = ['credit_score','credit_mix', 'payment_behaviour']
fig, axes = plt.subplots(1,3, figsize = (18,5), sharey = True)
color_score = {'Standard': 'green', 'Poor': 'red', 'Bad': 'red', 'Good': 'mediumblue'}
# Create bar charts
credit_score_count = df['credit_score'].value_counts().sort_values().reset_index()
sns.barplot(data = credit_score_count, y = 'count', x = 'credit_score', ax = axes[0], hue = 'credit_score', palette = color_score)

credit_mix_count = df['credit_mix'].value_counts().sort_values().reset_index()
sns.barplot(data = credit_mix_count, y = 'count', x = 'credit_mix', ax = axes[1], hue = 'credit_mix', palette = color_score)

df_payment_group = df.groupby(['payment_level','payment_volume']).size().reset_index(name = 'count')
sns.barplot(data = df_payment_group, x = 'payment_level', y = 'count', hue = 'payment_volume', ax = axes[2], palette = 'ocean')

# Set labels and display grouped tables for charts
ordinal_grouped_count_table = [credit_score_count, credit_mix_count, df_payment_group]

for ax, col, table in zip(axes,ordinal_cols, ordinal_grouped_count_table):
    ax.set_axisbelow(True)
    ax.set_title(f'Bar charts of {col}')
    print(f'Value counts for column {col}')
    display(table)

plt.legend(loc = 'upper left')
plt.tight_layout()
plt.show()

#### Explore about quantitative variables

In [ ]:
# Statistical outliers (1.5 x IQR rule) on the cleaned data
dis_ratio_var = ['age', 'num_bank_accounts','num_credit_card','num_of_loan','delay_from_due_date','num_of_delayed_payment',
                    'num_credit_inquiries','credit_history_age' ]

con_ratio_var = ['annual_income','monthly_inhand_salary','interest_rate','changed_credit_limit','outstanding_debt',
                'credit_utilization_ratio','total_emi_per_month','amount_invested_monthly','monthly_balance']

numeric_cols = dis_ratio_var + con_ratio_var
rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25,0.75])
    iqr = q3 - q1
    low, high = q1 - iqr*1.5, q3 + iqr*1.5
    n_outlier = int((df[col] <= low | df[col] >= high).sum())
    rows.append([col, round(iqr,2), round(low,2), round(high,2), n_outlier, 
                round((n_outlier/len(df))*100,1), df[col].min(), df[col].max()])

outlier_table = pd.DataFrame(rows, columns = ["Features", "IQR width", "(1.5)IQR lower", "(1.5)IQR upper",
                                                "Outliers", "Outliers (%)", "Min", "Max"]).set_index("Feature")

In [ ]:
# Draw histogram for 8 discrete ratio variables
fig, axes = plt.subplots(4,2,figsize = (20,20))
axes = axes.flatten()

for var, ax in zip(dis_ratio_var, axes):
    include_num = 'num' in var.lower()
    kde_status = False if include_num else True
    color =  'tab:blue' if include_num else 'tab:orange'
    sns.histplot(data = df, x = var, kde = kde_status, stat = 'count', discrete = True, color = color, kde_kws = {'bw_adjust': 0.7}, edgecolor = 'w', linewidth = 0.5, ax = ax)
    ax.set_title(f'Histogram of {var}')

plt.tight_layout()

# Draw histogram for 9 continuous ratio variables
fig, axes = plt.subplots(3,3,figsize = (20,20))
axes = axes.flatten()

for var, ax in zip(con_ratio_var, axes):
    sns.histplot(data = df, x = var, kde = True, stat = 'count', kde_kws = {'bw_adjust':0.7}, color = 'tab:green', edgecolor = 'w', linewidth = 0.5, ax = ax)
    ax.set_title(f'Histogram of {var}')

plt.tight_layout()
plt.show()

In [ ]:
boxplot_var = ['annual_income','outstanding_debt', 'credit_history_age', 'delay_from_due_date',
                'credit_utilization_ratio','total_emi_per_month']

fig, axes = plt.subplots(3, 2, figsize = (20,20))
axes = axes.flatten()

for var, ax in zip(boxplot_var, axes):
    sns.boxplot(data = df, x = 'credit_score', y = var, palette = 'Set2', hue = 'credit_score', ax = ax)
    ax.set_title(f'Boxplot of {var}, categorized by Credit Score')

plt.tight_layout()
plt.show()


In [ ]:
df = df.to_csv("my_data.csv", index=False)

## Conclusion after conducting EDA:
### Data Integrity and Class Distribution
- The dataset has no null variables or missing values.
- The target variable (`credit_score`) does not have extreme class imbalance; the "Standard" class has the highest number of records (~53.2%).
### Exploring variables
- Occupation distributions are fairly balanced across observers (ranging between 5% and 7.3% of the population), with Lawyers forming the largest group.
- `payment_of_min_amount` appears to be a major discriminator for credit scores. In the `No` group, 86.6% of observations maintain a `Standard` to `Good` credit score, with only 13.4% classified as `Poor`. However, for customers paying the minimum amount (`Yes` group), the proportion of `Good` credit scores sharply drops from 38.5% to 3.7%, while the proportion of `Poor` credit scores increases to 39.6%. This shift indicates that `payment_of_min_amount = Yes` serves as a red flag for potential credit risks.
- For payment behaviour, low-level spenders tend to make small-value payments, whereas high-level spenders mostly make medium-value payments.
- Age mostly ranges between 20 and 50 years old with a fairly equal distribution across records.
- `total_emi_per_month`, `amount_invested_monthly`, and `monthly_balance` appear right-skewed based on their histograms. Other variables exhibit multimodal distributions, which require deeper exploration using further statistical techniques or feature transformations.
- Boxplot analysis across `credit_score` classes reveals that `credit_utilization_ratio` is virtually identical across all target classes, suggesting it may have little to no predictive power for classification.
